# Week 2 Assignment — UAPOML
### Portfolio Management & Machine Learning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Generating Synthetic Price Data
Since we don't have live API access, I'm simulating realistic daily prices for 4 assets using geometric Brownian motion — same model used in Black-Scholes.

In [ ]:
# simulate 2 years of daily prices for 4 assets
n_days = 504
tickers = ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY']
S0 = np.array([2500, 3400, 1600, 1450])   # starting prices
mu = np.array([0.0004, 0.0003, 0.0002, 0.00035])  # daily drift
sigma = np.array([0.018, 0.016, 0.014, 0.017])    # daily vol

dates = pd.date_range('2022-01-03', periods=n_days, freq='B')
price_matrix = np.zeros((n_days, 4))
price_matrix[0] = S0

for t in range(1, n_days):
    z = np.random.randn(4)
    price_matrix[t] = price_matrix[t-1] * np.exp((mu - 0.5*sigma**2) + sigma*z)

prices_df = pd.DataFrame(price_matrix, index=dates, columns=tickers)
print(prices_df.tail())

---
## Section A — Portfolio Fundamentals & Risk Metrics

### P1 — Portfolio Construction, Daily Value & Annualised Volatility

In [ ]:
initial_capital = 1_000_000   # 10 lakh
weights = np.array([0.30, 0.25, 0.25, 0.20])  # must sum to 1

# capital allocated to each asset
allocation = initial_capital * weights

# units bought at first day prices
first_day_prices = prices_df.iloc[0].values
units = allocation / first_day_prices

print("Capital Allocation:")
for i, t in enumerate(tickers):
    print(f"  {t}: ₹{allocation[i]:,.0f}  |  units = {units[i]:.4f}")

# daily portfolio value — vectorised dot product
portfolio_val = prices_df.dot(units)

# daily returns
port_returns = portfolio_val.pct_change().dropna()

# annualised volatility
daily_vol = port_returns.std()
ann_vol = daily_vol * np.sqrt(252)
print(f"\nAnnualised Volatility: {ann_vol:.4f}  ({ann_vol*100:.2f}%)")

plt.figure(figsize=(11, 4))
portfolio_val.plot(color='steelblue', linewidth=1.4)
plt.title('Portfolio Value Over Time')
plt.xlabel('Date')
plt.ylabel('Portfolio Value (₹)')
plt.tight_layout()
plt.show()

### P2 — VaR (95% & 99%), CVaR, Max Drawdown

In [ ]:
returns = port_returns.values

# VaR — historical simulation
VaR_95 = np.percentile(returns, 5)
VaR_99 = np.percentile(returns, 1)

# CVaR — average of losses beyond VaR
CVaR_95 = returns[returns <= VaR_95].mean()
CVaR_99 = returns[returns <= VaR_99].mean()

# Max Drawdown
cumulative = (1 + port_returns).cumprod()
rolling_max = cumulative.cummax()
drawdown = (cumulative - rolling_max) / rolling_max
max_dd = drawdown.min()

print(f"VaR  (95%): {VaR_95*100:.3f}%")
print(f"VaR  (99%): {VaR_99*100:.3f}%")
print(f"CVaR (95%): {CVaR_95*100:.3f}%")
print(f"CVaR (99%): {CVaR_99*100:.3f}%")
print(f"Max Drawdown: {max_dd*100:.3f}%")

# chart with shaded tail
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax1 = axes[0]
ax1.hist(returns, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(VaR_95, color='orange', linestyle='--', label=f'VaR 95% ({VaR_95*100:.2f}%)')
ax1.axvline(VaR_99, color='red', linestyle='--', label=f'VaR 99% ({VaR_99*100:.2f}%)')
tail_vals = returns[returns <= VaR_95]
ax1.hist(tail_vals, bins=20, color='red', alpha=0.45, label='Tail (beyond VaR 95%)')
ax1.set_title('Return Distribution with VaR')
ax1.set_xlabel('Daily Return')
ax1.legend(fontsize=8)

ax2 = axes[1]
drawdown.plot(ax=ax2, color='crimson', linewidth=1.2)
ax2.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, color='crimson')
ax2.set_title('Portfolio Drawdown')
ax2.set_xlabel('Date')
ax2.set_ylabel('Drawdown')

plt.tight_layout()
plt.show()

### P3 — Sharpe & Sortino Ratio Comparison

In [ ]:
rf_annual = 0.065   # risk-free rate ~6.5% (approx RBI repo)
rf_daily = rf_annual / 252

mean_return = port_returns.mean()
total_vol = port_returns.std()

# downside deviation (only negative excess returns)
excess = port_returns - rf_daily
downside = excess[excess < 0]
downside_std = np.sqrt((downside**2).mean())

sharpe  = (mean_return - rf_daily) / total_vol * np.sqrt(252)
sortino = (mean_return - rf_daily) / downside_std * np.sqrt(252)

print(f"Sharpe Ratio : {sharpe:.4f}")
print(f"Sortino Ratio: {sortino:.4f}")
print()
if sortino > sharpe:
    print("Sortino > Sharpe — the portfolio's volatility is mostly on the upside, which is good.")
else:
    print("Sharpe > Sortino — downside volatility is relatively high.")

# individual asset Sharpe comparison
asset_returns = prices_df.pct_change().dropna()
asset_sharpe = {}
for col in tickers:
    r = asset_returns[col]
    s = (r.mean() - rf_daily) / r.std() * np.sqrt(252)
    asset_sharpe[col] = s

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(list(asset_sharpe.keys()), list(asset_sharpe.values()),
              color=['steelblue' if v > 0 else 'salmon' for v in asset_sharpe.values()])
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Annualised Sharpe Ratio — Individual Assets')
ax.set_ylabel('Sharpe Ratio')
for bar, v in zip(bars, asset_sharpe.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{v:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

### P4 — SMA Crossover Backtest

In [ ]:
# using RELIANCE as the asset to trade
asset_price = prices_df['RELIANCE'].copy()

sma10 = asset_price.rolling(10).mean()
sma30 = asset_price.rolling(30).mean()

# signal: +1 when SMA10 > SMA30, else -1
signal = pd.Series(np.where(sma10 > sma30, 1, -1), index=asset_price.index)
signal = signal.shift(1)   # avoid look-ahead bias

asset_ret = asset_price.pct_change()
strategy_ret = signal * asset_ret
strategy_ret = strategy_ret.dropna()

# portfolio value
strat_val = (1 + strategy_ret).cumprod() * initial_capital
bnh_val   = (1 + asset_ret.dropna()).cumprod() * initial_capital

# Win Rate & Profit Factor
trades = strategy_ret[strategy_ret != 0]
wins   = trades[trades > 0]
losses = trades[trades < 0]

win_rate = len(wins) / len(trades) if len(trades) > 0 else 0
pf = wins.sum() / abs(losses.sum()) if len(losses) > 0 else np.inf

strat_sharpe = (strategy_ret.mean() - rf_daily) / strategy_ret.std() * np.sqrt(252)

print(f"Total Trades    : {len(trades)}")
print(f"Win Rate        : {win_rate*100:.2f}%")
print(f"Profit Factor   : {pf:.3f}")
print(f"Strategy Sharpe : {strat_sharpe:.3f}")

plt.figure(figsize=(11, 4))
strat_val.plot(label='SMA Crossover Strategy', color='steelblue', linewidth=1.4)
bnh_val.plot(label='Buy & Hold', color='orange', linewidth=1.4, linestyle='--')
plt.title('SMA Crossover Strategy vs Buy & Hold (RELIANCE)')
plt.xlabel('Date')
plt.ylabel('Portfolio Value (₹)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Section B — Machine Learning Applications

### P5 — Feature Engineering & Min-Max Scaling (from scratch)

In [ ]:
def build_features(price_series):
    df = pd.DataFrame()
    df['Return_1d']    = price_series.pct_change()
    df['SMA_5']        = price_series.rolling(5).mean()
    df['SMA_20']       = price_series.rolling(20).mean()
    df['Volatility_10']= price_series.pct_change().rolling(10).std()
    df['Momentum_5']   = price_series - price_series.shift(5)
    return df

# build features for RELIANCE
feat_df = build_features(prices_df['RELIANCE'])
feat_df = feat_df.dropna().reset_index(drop=True)

# target: 1 if next day return > 0, else 0
feat_df['Target'] = (feat_df['Return_1d'].shift(-1) > 0).astype(int)
feat_df = feat_df.dropna().reset_index(drop=True)

feature_cols = ['Return_1d', 'SMA_5', 'SMA_20', 'Volatility_10', 'Momentum_5']
X = feat_df[feature_cols].values
y = feat_df['Target'].values

# Min-Max Scaling — from scratch
def minmax_scale(X):
    x_min = X.min(axis=0)
    x_max = X.max(axis=0)
    return (X - x_min) / (x_max - x_min + 1e-8), x_min, x_max

X_scaled, x_min, x_max = minmax_scale(X)

print("Feature matrix shape:", X_scaled.shape)
print("Scaled range check (min/max per feature):")
for i, col in enumerate(feature_cols):
    print(f"  {col}: [{X_scaled[:, i].min():.3f}, {X_scaled[:, i].max():.3f}]")

print("\nFirst 5 rows of scaled features:")
print(pd.DataFrame(X_scaled[:5], columns=feature_cols).round(4))

### P6 — KNN Classifier (from scratch) — Euclidean Distance, k-tuning, Confusion Matrix

In [ ]:
# train-test split — 80/20, keeping time order
split = int(0.8 * len(X_scaled))
X_train, X_test = X_scaled[:split], X_scaled[split:]
y_train, y_test = y[:split], y[split:]

def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

def knn_predict(X_train, y_train, X_test, k):
    preds = []
    for xq in X_test:
        dists = [euclidean_distance(xq, xt) for xt in X_train]
        k_idx = np.argsort(dists)[:k]
        k_labels = y_train[k_idx]
        preds.append(np.bincount(k_labels.astype(int)).argmax())
    return np.array(preds)

# test across different k values
k_values = [3, 5, 7, 11, 15]
accuracies = []

for k in k_values:
    y_pred = knn_predict(X_train, y_train, X_test, k)
    acc = np.mean(y_pred == y_test)
    accuracies.append(acc)
    print(f"k={k:2d}  Accuracy: {acc*100:.2f}%")

best_k = k_values[np.argmax(accuracies)]
print(f"\nBest k = {best_k} with accuracy {max(accuracies)*100:.2f}%")

plt.figure(figsize=(7, 4))
plt.plot(k_values, [a*100 for a in accuracies], marker='o', color='steelblue', linewidth=2)
plt.axvline(best_k, linestyle='--', color='orange', label=f'Best k={best_k}')
plt.xlabel('k')
plt.ylabel('Accuracy (%)')
plt.title('KNN Accuracy vs k')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix + Precision & Recall for best k
y_best = knn_predict(X_train, y_train, X_test, best_k)

TP = np.sum((y_best == 1) & (y_test == 1))
TN = np.sum((y_best == 0) & (y_test == 0))
FP = np.sum((y_best == 1) & (y_test == 0))
FN = np.sum((y_best == 0) & (y_test == 1))

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

cm = np.array([[TN, FP], [FN, TP]])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
for (i, j), val in np.ndenumerate(cm):
    ax.text(j, i, str(val), ha='center', va='center', fontsize=14, fontweight='bold')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred 0', 'Pred 1'])
ax.set_yticklabels(['Actual 0', 'Actual 1'])
ax.set_title(f'Confusion Matrix (k={best_k})')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### P7 — Linear Regression: Normal Equation + Gradient Descent + Loss Curve

In [ ]:
# target: next day's actual return (regression, not classification)
feat_reg = build_features(prices_df['RELIANCE']).dropna()
feat_reg = feat_reg.reset_index(drop=True)
feat_reg['Next_Return'] = feat_reg['Return_1d'].shift(-1)
feat_reg = feat_reg.dropna().reset_index(drop=True)

X_reg = feat_reg[feature_cols].values
y_reg = feat_reg['Next_Return'].values

X_reg_s, xmin, xmax = minmax_scale(X_reg)

# add bias column
X_bias = np.hstack([np.ones((X_reg_s.shape[0], 1)), X_reg_s])

split_r = int(0.8 * len(X_bias))
X_tr, X_te = X_bias[:split_r], X_bias[split_r:]
y_tr, y_te = y_reg[:split_r], y_reg[split_r:]

# --- Method 1: Normal Equation ---
beta_ne = np.linalg.lstsq(X_tr, y_tr, rcond=None)[0]   # more numerically stable than direct inversion

y_pred_ne = X_te @ beta_ne
mse_ne = np.mean((y_te - y_pred_ne)**2)
ss_res = np.sum((y_te - y_pred_ne)**2)
ss_tot = np.sum((y_te - y_te.mean())**2)
r2_ne = 1 - ss_res / ss_tot

print("=== Normal Equation ===")
print(f"MSE : {mse_ne:.8f}")
print(f"R²  : {r2_ne:.4f}")

In [ ]:
# --- Method 2: Batch Gradient Descent ---
eta = 0.01
n_iters = 1000
n = len(X_tr)

beta_gd = np.zeros(X_tr.shape[1])
loss_history = []

for i in range(n_iters):
    y_hat = X_tr @ beta_gd
    residual = y_hat - y_tr
    grad = (1 / n) * X_tr.T @ residual
    beta_gd -= eta * grad
    loss = np.mean(residual**2)
    loss_history.append(loss)

y_pred_gd = X_te @ beta_gd
mse_gd = np.mean((y_te - y_pred_gd)**2)
r2_gd = 1 - np.sum((y_te - y_pred_gd)**2) / np.sum((y_te - y_te.mean())**2)

print("=== Gradient Descent ===")
print(f"MSE : {mse_gd:.8f}")
print(f"R²  : {r2_gd:.4f}")

print("\nLearned Coefficients (GD):")
coef_names = ['Bias'] + feature_cols
for name, coef in zip(coef_names, beta_gd):
    print(f"  {name}: {coef:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# loss curve
axes[0].plot(loss_history, color='steelblue', linewidth=1.2)
axes[0].set_title('Gradient Descent Loss Curve')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE')

# actual vs predicted
axes[1].scatter(y_te, y_pred_gd, alpha=0.4, color='steelblue', s=15)
lims = [min(y_te.min(), y_pred_gd.min()), max(y_te.max(), y_pred_gd.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1.2, label='Perfect Fit')
axes[1].set_title('Actual vs Predicted Returns (GD)')
axes[1].set_xlabel('Actual Return')
axes[1].set_ylabel('Predicted Return')
axes[1].legend()

plt.tight_layout()
plt.show()

### P8 — ML-Driven Allocation vs Equal-Weight Backtest

In [ ]:
# train a linear regression model for each asset, predict expected returns on test period
# then build weights proportional to max(predicted_return, 0)

predicted_returns = {}

for ticker in tickers:
    f = build_features(prices_df[ticker]).dropna().reset_index(drop=True)
    f['Next_Return'] = f['Return_1d'].shift(-1)
    f = f.dropna().reset_index(drop=True)

    Xr = f[feature_cols].values
    yr = f['Next_Return'].values
    Xr_s, _, _ = minmax_scale(Xr)
    Xr_b = np.hstack([np.ones((Xr_s.shape[0], 1)), Xr_s])

    spl = int(0.8 * len(Xr_b))
    beta = np.linalg.lstsq(Xr_b[:spl], yr[:spl], rcond=None)[0]
    # predicted returns on test set
    predicted_returns[ticker] = Xr_b[spl:] @ beta

# align lengths
min_len = min(len(v) for v in predicted_returns.values())
mu_hat = np.array([predicted_returns[t][:min_len] for t in tickers]).T  # shape: (test_days, 4)

# daily weights: positive allocations only, normalised
daily_weights = np.maximum(mu_hat, 0)
weight_sums = daily_weights.sum(axis=1, keepdims=True)
# if all negative, fall back to equal weight
mask = (weight_sums == 0).flatten()
daily_weights[mask] = 0.25
weight_sums[mask] = 1.0
daily_weights = daily_weights / weight_sums

# get actual returns for test period
asset_ret_df = prices_df[tickers].pct_change().dropna()
n_train_rows = int(0.8 * (len(asset_ret_df) - 1))
test_ret = asset_ret_df.iloc[n_train_rows : n_train_rows + min_len].values

# ML portfolio return each day
ml_daily_ret  = (daily_weights * test_ret).sum(axis=1)
eq_daily_ret  = test_ret.mean(axis=1)

ml_val = (1 + ml_daily_ret).cumprod() * initial_capital
eq_val = (1 + eq_daily_ret).cumprod() * initial_capital

ml_sharpe = (ml_daily_ret.mean() - rf_daily) / (ml_daily_ret.std() + 1e-8) * np.sqrt(252)
eq_sharpe = (eq_daily_ret.mean() - rf_daily) / (eq_daily_ret.std() + 1e-8) * np.sqrt(252)

print(f"ML-Driven Portfolio  | Final Value: ₹{ml_val[-1]:,.0f} | Sharpe: {ml_sharpe:.3f}")
print(f"Equal-Weight         | Final Value: ₹{eq_val[-1]:,.0f} | Sharpe: {eq_sharpe:.3f}")

plt.figure(figsize=(11, 4))
plt.plot(ml_val, label='ML-Driven Allocation', color='steelblue', linewidth=1.5)
plt.plot(eq_val, label='Equal Weight', color='orange', linewidth=1.5, linestyle='--')
plt.title('ML-Driven vs Equal-Weight Portfolio (Test Period)')
plt.xlabel('Days')
plt.ylabel('Portfolio Value (₹)')
plt.legend()
plt.tight_layout()
plt.show()

### P9 — 5-Fold Cross-Validation & Model Comparison Table

In [ ]:
# 5-fold CV for KNN
def kfold_cv_knn(X, y, k, n_folds=5):
    fold_size = len(X) // n_folds
    accs = []
    for fold in range(n_folds):
        val_start = fold * fold_size
        val_end   = val_start + fold_size
        X_val = X[val_start:val_end]
        y_val = y[val_start:val_end]
        X_tr  = np.concatenate([X[:val_start], X[val_end:]], axis=0)
        y_tr  = np.concatenate([y[:val_start], y[val_end:]], axis=0)
        preds = knn_predict(X_tr, y_tr, X_val, k)
        accs.append(np.mean(preds == y_val))
    return np.mean(accs), np.std(accs)

def kfold_cv_linreg(X, y, n_folds=5):
    fold_size = len(X) // n_folds
    r2s = []
    for fold in range(n_folds):
        val_start = fold * fold_size
        val_end   = val_start + fold_size
        X_val = X[val_start:val_end]
        y_val = y[val_start:val_end]
        X_tr  = np.concatenate([X[:val_start], X[val_end:]], axis=0)
        y_tr  = np.concatenate([y[:val_start], y[val_end:]], axis=0)
        beta  = np.linalg.lstsq(X_tr, y_tr, rcond=None)[0]
        y_hat = X_val @ beta
        ss_res = np.sum((y_val - y_hat)**2)
        ss_tot = np.sum((y_val - y_val.mean())**2)
        r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0
        r2s.append(r2)
    return np.mean(r2s), np.std(r2s)

# run CV
knn_mean, knn_std = kfold_cv_knn(X_scaled, y, k=best_k)
lr_mean,  lr_std  = kfold_cv_linreg(X_bias, y_reg)

print(f"KNN (k={best_k}) 5-Fold CV Accuracy : {knn_mean*100:.2f}% ± {knn_std*100:.2f}%")
print(f"LinReg 5-Fold CV R²            : {lr_mean:.4f} ± {lr_std:.4f}")

# Model comparison table
comparison = pd.DataFrame({
    'Model':       ['KNN Classifier', 'Linear Regression'],
    'Metric':      ['Accuracy',        'R²'],
    'CV Mean':     [f"{knn_mean*100:.2f}%", f"{lr_mean:.4f}"],
    'CV Std':      [f"{knn_std*100:.2f}%", f"{lr_std:.4f}"],
    'Best Param':  [f'k={best_k}',          'η=0.01, 1000 iters']
})
print("\nModel Comparison:")
print(comparison.to_string(index=False))

---
## Section C — Conceptual & Critical Thinking

### P10a — Diversification Math (Portfolio Variance)

For a two-asset portfolio with weights $w_1$, $w_2 = 1 - w_1$:

$$\sigma_p^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \rho_{12} \sigma_1 \sigma_2$$

When $\rho_{12} < 1$ (assets are not perfectly correlated), the portfolio variance is **strictly less** than the weighted average of individual variances. This is the mathematical basis of diversification — you can reduce risk without sacrificing the same proportion of return.

Extreme cases:
- $\rho = 1$: no diversification benefit, $\sigma_p = w_1\sigma_1 + w_2\sigma_2$
- $\rho = -1$: maximum diversification, variance can theoretically drop to zero
- $\rho = 0$: uncorrelated assets, variance reduces by approximately $1/n$ for equal-weight portfolio

That's why Markowitz called diversification the only free lunch — you're reducing risk without giving up expected return.

In [ ]:
# numerical illustration
s1, s2 = 0.20, 0.25   # annual volatility
w1 = 0.5; w2 = 0.5
rhos = np.linspace(-1, 1, 100)
port_var = w1**2 * s1**2 + w2**2 * s2**2 + 2*w1*w2*rhos*s1*s2

plt.figure(figsize=(8, 4))
plt.plot(rhos, np.sqrt(port_var) * 100, color='steelblue', linewidth=2)
plt.axhline(w1*s1*100 + w2*s2*100, linestyle='--', color='orange', label='Weighted avg vol (no diversification)')
plt.xlabel('Correlation (ρ)')
plt.ylabel('Portfolio Volatility (%)')
plt.title('Portfolio Volatility vs Correlation Between Assets')
plt.legend()
plt.tight_layout()
plt.show()

### P10b — Fundamental Features for KNN

If we had access to fundamental financial data, useful features for a KNN model would be:

1. **P/E Ratio** — measures how expensive a stock is relative to its earnings. Low P/E might indicate undervaluation.
2. **Debt-to-Equity (D/E)** — higher leverage means higher financial risk. KNN could cluster high-risk vs low-risk companies.
3. **Revenue Growth YoY** — captures business momentum, which may predict future price movement.
4. **Return on Equity (ROE)** — measures how efficiently management generates profit. High ROE companies tend to outperform.
5. **Free Cash Flow Yield** — indicates financial health; companies generating strong FCF tend to sustain growth.

All of these would need to be Min-Max scaled before feeding into KNN, since they have very different ranges. For example, P/E ratios can range from 5 to 100+, while D/E ratios sit between 0 and 3 for most companies. Without scaling, P/E would completely dominate the distance calculation.

### P10c — Curse of Dimensionality

As the number of features (dimensions) increases, the Euclidean distance between any two points in the dataset starts to converge — the nearest and farthest neighbours become almost equally distant. This breaks the core assumption of KNN, which relies on nearness being meaningful.

Mathematically, in high dimensions most of the data volume shifts to the "edges" of the space. If we have $n$ training points spread across $d$ dimensions, the effective density of points near any query point decreases exponentially with $d$.

In practice for financial ML:
- Adding too many features (RSI, MACD, multiple lag windows, fundamental ratios) can actually hurt KNN accuracy.
- Mitigation: feature selection (keep only the most informative features), PCA to reduce dimensionality while preserving variance, or use a distance metric like cosine similarity that handles high-dimensional spaces better.
- For the features in this assignment (5 features), we're in a manageable range, but it's worth monitoring performance as more features are added.

### P10d — Overfitting in Backtesting

Overfitting in backtesting happens when a strategy appears profitable only because it was tuned on the same data it's being evaluated on. The model (or strategy parameters) fit the noise in historical data rather than genuine patterns.

Common ways this happens:
- **Optimising SMA windows** over the full historical period and reporting that as the result. The windows 10/30 might have worked for 2020–2022 but fail in different market regimes.
- **Data snooping bias**: testing 50 different indicator combinations and picking the one that worked best. Even random strategies will occasionally look good by chance.
- **Look-ahead bias**: accidentally using future information (e.g. computing a 30-day SMA at day 15).

Mitigation:
- Strict train/validation/test split. Never touch the test set until final evaluation.
- Walk-forward testing: expand the training window, validate on the next block, and repeat. This mimics how a real strategy would be deployed.
- Reduce the number of free parameters in the strategy.

### P10e — Linear Regression Assumptions in Finance

Classical linear regression assumes:

1. **Linearity**: the relationship between features and the target is linear. In finance, return drivers are often non-linear (momentum can reverse, volatility clustering, etc.).
2. **Independence of errors**: residuals should not be autocorrelated. Financial returns exhibit autocorrelation, especially in volatility (ARCH effects).
3. **Homoscedasticity**: constant variance of errors. Stock returns have time-varying volatility (heteroscedasticity). High-vol periods produce larger errors than low-vol periods.
4. **Normality of residuals**: regression errors assumed Gaussian. Financial returns have fat tails — extreme moves happen far more often than a normal distribution predicts.
5. **No multicollinearity**: features should not be linearly dependent. SMA_5 and SMA_20 are both derived from the same price series, so they will be correlated.

Despite violating several assumptions, linear regression is still widely used as a baseline in quantitative finance because:
- It's interpretable — coefficients directly show feature importance and direction
- It's fast to train and avoids overfitting when the number of features is small relative to observations
- The violations don't make it useless, just mean that confidence intervals and p-values shouldn't be taken at face value